## Weighted stats

In [12]:
import pandas as pd

In [13]:
url = "https://ws.cso.ie/public/api.restful/PxStat.Data.Cube_API.ReadDataset/FY006A/CSV/1.0/en"
df = pd.read_csv(url)
df.tail(3)

,STATISTIC,Statistic Label,TLIST(A1),CensusYear,C02199V02655,Sex,C02076V03371,Single Year of Age,C03789V04537,Administrative Counties,UNIT,VALUE
9789,FY006AC01,Population,2022,2022,2,Female,650,100 years and over,2ae19629-149d-13a3-e055-000000000001,Cavan County Council,Number,12
9790,FY006AC01,Population,2022,2022,2,Female,650,100 years and over,2ae19629-14a4-13a3-e055-000000000001,Donegal County Council,Number,31
9791,FY006AC01,Population,2022,2022,2,Female,650,100 years and over,2ae19629-1495-13a3-e055-000000000001,Monaghan County Council,Number,7


Remove individual sex (i.e. Female, Male) leaving only Both sexes

In [14]:
df = df[df['Sex'] != 'Female']
df = df[df['Sex'] != 'Male']
df.tail(3)

,STATISTIC,Statistic Label,TLIST(A1),CensusYear,C02199V02655,Sex,C02076V03371,Single Year of Age,C03789V04537,Administrative Counties,UNIT,VALUE
3261,FY006AC01,Population,2022,2022,-,Both sexes,650,100 years and over,2ae19629-149d-13a3-e055-000000000001,Cavan County Council,Number,18
3262,FY006AC01,Population,2022,2022,-,Both sexes,650,100 years and over,2ae19629-14a4-13a3-e055-000000000001,Donegal County Council,Number,33
3263,FY006AC01,Population,2022,2022,-,Both sexes,650,100 years and over,2ae19629-1495-13a3-e055-000000000001,Monaghan County Council,Number,8


### Prepare data for analysis 
Drop unnecesarry colums

In [15]:
headers = df.columns.tolist
headers

<bound method IndexOpsMixin.tolist of Index(['STATISTIC', 'Statistic Label', 'TLIST(A1)', 'CensusYear',
       'C02199V02655', 'Sex', 'C02076V03371', 'Single Year of Age',
       'C03789V04537', 'Administrative Counties', 'UNIT', 'VALUE'],
      dtype='object')>

In [16]:
drop_col_list = ['STATISTIC', 'Statistic Label','TLIST(A1)','CensusYear','C02199V02655','Sex','C02076V03371','C03789V04537','UNIT']
df.drop(columns=drop_col_list, inplace=True)
print (df.head(3))

  Single Year of Age Administrative Counties    VALUE
0           All ages                 Ireland  5149139
1           All ages   Carlow County Council    61968
2           All ages     Dublin City Council   592713


In [17]:
# Remove All ages rows
df = df[df["Single Year of Age"] != 'All ages']
print (df.head(3))
# Replace "Under 1 year" with "0"
df["Single Year of Age"] = df["Single Year of Age"].str.replace("Under 1 year","0")
# Replace non-digit characters with an empty string
df["Single Year of Age"] = df["Single Year of Age"].str.replace("\D", "", regex=True)

   Single Year of Age Administrative Counties  VALUE
32       Under 1 year                 Ireland  57796
33       Under 1 year   Carlow County Council    699
34       Under 1 year     Dublin City Council   6213


In [18]:
# Convert the "Single Year of Age" column to integer type for further analysis
df["Single Year of Age"] = df["Single Year of Age"].astype(int)
print (df.head(3))
df.info()

    Single Year of Age Administrative Counties  VALUE
32                   0                 Ireland  57796
33                   0   Carlow County Council    699
34                   0     Dublin City Council   6213
<class 'pandas.core.frame.DataFrame'>
Index: 3232 entries, 32 to 3263
Data columns (total 3 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Single Year of Age       3232 non-null   int64 
 1   Administrative Counties  3232 non-null   object
 2   VALUE                    3232 non-null   int64 
dtypes: int64(2), object(1)
memory usage: 101.0+ KB


In [19]:
# Configure data for analysis using a pivot table
df_analysis = pd.pivot_table(df, 'VALUE',"Single Year of Age","Administrative Counties")
print (df_analysis.head(3))

df_analysis.to_csv("population_for_analysis.csv")

Administrative Counties  Carlow County Council  Cavan County Council  \
Single Year of Age                                                     
0                                          699                  1006   
1                                          649                  1004   
2                                          689                  1061   

Administrative Counties  Clare County Council  Cork City Council  \
Single Year of Age                                                 
0                                        1377               2283   
1                                        1360               2284   
2                                        1473               2306   

Administrative Counties  Cork County Council  Donegal County Council  \
Single Year of Age                                                     
0                                       4190                    1797   
1                                       4115                    1881   
2         

## Weighted Descriptive Statistics

`Weighted mean` = sum(age * population at that age) / sum(populations at age)

In [9]:
headers = df_analysis.columns.tolist()
district = headers[0]
district

'Carlow County Council'

Calculate the total number of people in Carlow using variable district pointing at Carlow column and method `.sum()`

In [10]:
number_of_people = df_analysis[district].sum()
number_of_people

61968

The below cell won't run, because the Single Year of age is an index, and county (district) starts with 0 column. Therefore the fix is in the following cell

In [11]:
cumulative_age = (df_analysis['Single Year of Age']*df_analysis[district]).sum()
print(cumulative_age)

KeyError: 'Single Year of Age'

In [ ]:
cumulative_age = df_analysis[district].mul(df_analysis.index, axis=0).sum()
cumulative_age

## Weighted Mean

In [ ]:
weighted_mean = cumulative_age / number_of_people
weighted_mean

## Weighted Median

Create a series of the cumlative sums and find the index of the middle value.

In [ ]:
# Cumulative sum
cumsum = df_analysis[district].cumsum()
cumsum

Sum = number of people

In [ ]:
# Find the sum and divide by 2 to get the cutoff value
cutoff = df_analysis[district].sum()/2
cutoff

In [ ]:
# Median >> looking where cumulative sum is greater than or equal to cutoff and returning an index (age) of it
df_analysis[district][cumsum>=cutoff].index[0]

## Weighted Standard Deviation

We can't use regular std() function for our dataset as the collumn contains number of populations for each age (index column). Doing mean() and std() on counts return wrong results - see two cells below:

In [ ]:
# Incorrect way 
df_analysis[district].std()

In [ ]:
# Incorrect way 
df_analysis[district].mean()

We can use `Numpy` to find weighted mean in a simpler way than we've done previously

In [ ]:
import numpy as np

w_mean = np.average(df_analysis.index, weights=df_analysis[district])
w_mean

Find Variance

In [ ]:
w_var = np.average((df_analysis.index - w_mean)**2, weights=df_analysis[district])
w_var

Now find Standard Deviation

In [ ]:
w_std = np.sqrt(w_var)
w_std